In [1]:
!pip install -q -U google-genai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 790.4/790.4 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.5/246.5 kB 15.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.50.0 which is incompatible.


In [2]:
import os, re, time
from collections import Counter
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found!")
client = genai.Client(api_key=api_key)
print("Gemini ready ✅")

Gemini ready ✅


In [3]:
DICTIONARY_PATH  = "Dictionary.txt"
WORD_LIST_PATH   = "final_word_based_corpus.txt"
COMMON_FREQ_PATH = "wordfreq_si_clean.txt"
CORPUS_PATH      = "sinhala_large_corpus.txt"

In [4]:
SINHALA_RE = re.compile(r"[\u0D80-\u0DFF]+")

def extract_sinhala_words(text):
    return SINHALA_RE.findall(text)

def load_word_list(path):
    s = set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            s.update(extract_sinhala_words(line))
    return s

def load_frequency_words(path):
    d = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2 and parts[1].isdigit():
                d[parts[0]] = int(parts[1])
    return d

def load_corpus_words(path):
    c = Counter()
    with open(path, encoding="utf-8") as f:
        for line in f:
            c.update(extract_sinhala_words(line))
    return dict(c)

dictionary_words       = load_word_list(DICTIONARY_PATH)
final_word_words       = load_word_list(WORD_LIST_PATH)
common_frequency_words = load_frequency_words(COMMON_FREQ_PATH)
corpus_frequency_words = load_corpus_words(CORPUS_PATH)

combined_frequency = Counter()
combined_frequency.update(common_frequency_words)
combined_frequency.update(corpus_frequency_words)

valid_words = (
    dictionary_words | final_word_words
    | set(common_frequency_words) | set(corpus_frequency_words)
)
print(f"Total valid words: {len(valid_words)}")

Total valid words: 51755


In [5]:
def has_english(text):
    return bool(re.search(r"[A-Za-z]", text))

def clean_spaces(text):
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.!?།])", r"\1", text)
    return text.strip()

def sinhala_clean(text):
    text = re.sub(r"[*_`#>~]", "", text)
    text = re.sub(r"[A-Za-z]", "", text)
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    return clean_spaces(" ".join(lines))

def get_unknown_words(text):
    return sorted(set(w for w in extract_sinhala_words(text) if w not in valid_words))

def get_low_freq_words(text, min_score=2):
    return sorted(
        [(w, combined_frequency.get(w, 0))
         for w in set(extract_sinhala_words(text))
         if combined_frequency.get(w, 0) < min_score],
        key=lambda x: x[1]
    )

In [6]:
# ─── Domain-specific phonetic glossary ────────────────────────────────────────
# Pre-processes the worst phonetic clusters BEFORE the AI sees the text.
# Keys are exact phonetic forms found in real technician speech.
# This runs BEFORE every API call so the model starts from a cleaner base.
PHONETIC_GLOSSARY = {
    # Service names
    "පියෝටීවිසහ":    "පියෝ ටීවී සහ",
    "පියෝටීවි":      "පියෝ ටීවී",
    "ඔයිස්":         "වොයිස්",
    "ටවඉස්":         "ට්වයිස්",

    # Line status
    "සමොකද්වෙලා":   "සම්බන්ධ වෙලා",
    "කපල":           "කපලා",
    "හරිමේ":         "හරි. මේ",
    "කන්දක":         "කොටසක",
    "කෙසර":          "කිසෙක",

    # Numbers / billing
    "රනාඩෙදාස්":    "රුපියල් දෙදහස්",
    "පන්සේ":         "පන්සිය",
    "අසුතුනත්ත":     "අසූ තුනක්",
    "සර්ගෙවන්නතියනැච්චර": "සර් ගෙවන්න තියෙනවා, ඊට",
    "ගවන්නතියෙන්නේවසසෙ": "ගෙවන්න තියෙන්නේ, ඔය",

    # Cable work
    "ගලවනානේ":      "ගලවනවා නේ",
    "කළුපාඩ":        "කළු පාට",
    "නිට්":          "නෙට්",
    "නීට්":          "නෙට්",
    "නවා":           "දාලා",
    "ගේලක":          "ගේමක",
    "හතරෙන්නේ":      "හවරෙන්",
    "කවුලලිව":       "කවරය",
    "හමට":           "ගමට",
    "ගලට":           "ගලවා",

    # Closing / status
    "පශික්කරනකන්":   "පරීක්ෂා කරනකන්",
    "හරිසහරිවකස":    "හරි. සහ හරිම",
    "එසටමත":         "ඒ සඳහාම",
    "සවදස්":         "සුබ දවසක්",
    "සබදාසක්":       "ඇමතුමේ",
    "ඇමතිමකිසනා":    "ඇමතුමේ රැඳී ඉන්න",
    "රඳඉන්න":        "රැඳී ඉන්න",
    "වහිනකොට":       "වහිනකොට",   # keep — contextually OK
}

def apply_glossary(text):
    for wrong, correct in PHONETIC_GLOSSARY.items():
        text = text.replace(wrong, correct)
    return text

In [7]:
MODEL = "models/gemini-2.5-flash"

# ─── System prompt — no gold example, pure domain knowledge ───────────────────
SYSTEM_PROMPT = """
You are an expert Sinhala language editor who specialises in Sri Lankan ISP
(internet service provider) customer support call transcripts.

SCENARIO: A field technician or call-centre agent is speaking colloquial Sinhala
about a customer's fiber-optic internet connection. Their speech was typed roughly
(phonetically) and contains errors. You must reconstruct the natural, correct
Sinhala that they intended to say.

━━━ WHAT THE CALLS ARE ABOUT ━━━
• Checking whether a customer's line is active ("ඔන්") or cut ("කපා")
• Confirming which services are working: Peo TV (පියෝ ටීවී), Voice/VoIP (වොයිස්),
  broadband internet
• Quoting outstanding bill amounts in Sri Lankan rupees
• Advising customers to check cables, boxes, connectors
• Closing calls politely: "සුබ දවසක්, ඇමතුමේ රැඳී ඉන්න"

━━━ COMMON PHONETIC PATTERNS TO FIX ━━━
Phonetic input          → Correct Sinhala
──────────────────────────────────────────
"කපල"                  → "කපලා"
"ඔයිස්" / "ඕයිස්"     → "වොයිස්"
"නිට්" / "නීට්"        → "නෙට්"  (network cable connector)
"ගේලක"                 → "ගේමක"  (informal: holder/port)
"ගලවනානේ"              → "ගලවනවා නේ"
"කළුපාඩ"               → "කළු පාට"
"සමොකද්"               → "සම්බන්ධ"
"රනාඩෙදාස් පන්සේ අසුතුනත්ත" → "රුපියල් දෙදහස් පන්සිය අසූ තුනක්"
"සවදස්"                → "සුබ දවසක්"
"සබදාසක් ස ඇමතිමකිසනා රඳඉන්න" → "ඇමතුමේ රැඳී ඉන්න"
merged words like "වැඩකරනා" → "වැඩ කරනා"
merged phrases like "හරිමේලයින්" → "හරි. මේ ලයින්"

━━━ TECHNICAL TERMS (keep in Sinhala script) ━━━
ලයින්, කේබල්, නෙට් කේබල්, ෆයිබර්, පියෝ ටීවී, වොයිස්, ඔන්, ඔෆ්,
බොක්ස්, රවුටර්, ස්ප්ලිටර්, ස්ප්ලිසර්, ජොයින්ට්, ටේප්, ෆේස්, ස්විච්

━━━ STRICT OUTPUT RULES ━━━
1. Output ONLY the corrected Sinhala — no labels, no explanations, no markdown.
2. Keep ALL content — nothing omitted, nothing reordered.
3. Colloquial forms are correct: "නෑ", "ඇති", "නේ", "ඔව්", "එච්චරයි"
4. One clean paragraph.
5. Only Sinhala Unicode (U+0D80–U+0DFF), spaces, digits, and punctuation.
"""

# ─── Generic few-shot examples (no gold example) ──────────────────────────────
FEW_SHOT = [
    {
        "input":  "ලයින් ඔන්එකෙ තියෙනවා කපලනෑ නෙට් කේබල් හරිනෑ",
        "output": "ලයින් ඔන් එකේ තියෙනවා, කපලා නෑ. නෙට් කේබල් හරි නෑ."
    },
    {
        "input":  "පියෝටීවිකත් ඔන් ඔයිස් එකත් ඔන් ගැටලුවක්නෑ",
        "output": "පියෝ ටීවී එකත් ඔන්, වොයිස් එකත් ඔන්, ගැටලුවක් නෑ."
    },
    {
        "input":  "රුපියල් දෙදහස් පන්සිය ගෙවන්නතියෙනවා සරයි ගෙවන්නතියෙන්නේ",
        "output": "රුපියල් දෙදහස් පන්සිය ගෙවන්න තියෙනවා, ඒ සරයි ගෙවන්න තියෙන්නේ."
    },
    {
        "input":  "කළුපාඩ බොක්සෙන් නිට් කේබල් ගැලිලා ඇති පරීක්ෂා කරන්නකො",
        "output": "කළු පාට බොක්ස් එකෙන් නෙට් කේබල් ගැලිලා ඇති, පරීක්ෂා කරන්නකෝ."
    },
    {
        "input":  "ඔක්කොම හරි සුබදවසක් ඇමතුමේ රඳඉන්න",
        "output": "ඔක්කොම හරි. සුබ දවසක්, ඇමතුමේ රැඳී ඉන්න."
    },
]

def build_prompt(text, mode="correct", unknown_words=None):
    shots = "\n\n".join(
        f"Input: {s['input']}\nOutput: {s['output']}" for s in FEW_SHOT
    )
    base = SYSTEM_PROMPT.strip() + "\n\n=== EXAMPLES ===\n" + shots + "\n\n"

    if mode == "correct":
        return base + f"=== CORRECT THIS TEXT ===\nInput: {text}\nOutput:"
    elif mode == "refine":
        return (
            base
            + "=== REFINE — fix any remaining merged words, wrong technical terms, "
            "or broken syllables. Output corrected Sinhala ONLY ===\n"
            f"Text: {text}\nCorrected:"
        )
    else:  # targeted
        wlist = ", ".join(unknown_words or [])
        return (
            base
            + f"=== TARGETED FIX — these words look wrong: {wlist}\n"
            "Fix each one in context. Output the full corrected paragraph ONLY ===\n"
            f"Text: {text}\nCorrected:"
        )

In [8]:
def call_model(prompt, temperature=0.1, thinking_budget=10000, max_retries=3):
    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=temperature,
                    top_p=0.92,
                    max_output_tokens=8000,
                    thinking_config=types.ThinkingConfig(
                        thinking_budget=thinking_budget
                    )
                )
            )
            raw = resp.text.strip()
            raw = re.sub(
                r"^(Output|Corrected)(\s+text)?\s*:\s*", "",
                raw, flags=re.IGNORECASE
            ).strip()
            cleaned = sinhala_clean(raw)
            if cleaned and not has_english(cleaned):
                return cleaned
        except Exception as e:
            wait = 2 ** attempt
            print(f"  Attempt {attempt+1} failed: {e}. Retrying in {wait}s…")
            time.sleep(wait)
    return None


def correct_with_gemini(text):
    """
    Pipeline:
      Pre-process  → fix worst phonetic clusters with the glossary
      Pass 1       → full correction (high thinking budget)
      Pass 2       → self-review / refinement
      Pass 3       → targeted fix of any still-unknown words
    """
    if not text.strip():
        return ""

    preprocessed = apply_glossary(text)
    print("▶ Pre-processed input ready.")

    print("▶ Pass 1: Full correction…")
    p1 = call_model(build_prompt(preprocessed, mode="correct"),
                    temperature=0.1, thinking_budget=10000)
    if not p1:
        return "[ERROR: Pass 1 failed]"
    print(f"  ✓ Pass 1 done.")

    print("▶ Pass 2: Refinement…")
    p2 = call_model(build_prompt(p1, mode="refine"),
                    temperature=0.0, thinking_budget=5000)
    result = p2 if p2 else p1
    print(f"  ✓ Pass 2 done.")

    unknown = get_unknown_words(result)
    if unknown:
        print(f"▶ Pass 3: Fixing unknown words: {unknown}")
        p3 = call_model(build_prompt(result, mode="targeted", unknown_words=unknown),
                        temperature=0.0, thinking_budget=5000)
        if p3:
            result = p3
        print(f"  ✓ Pass 3 done.")
    else:
        print("▶ Pass 3: Skipped — no unknown words ✅")

    return result

In [9]:
def correct_sinhala_text(text):
    corrected = correct_with_gemini(text)
    return {
        "input":               text,
        "corrected_text":      corrected,
        "unknown_words":       get_unknown_words(corrected),
        "low_frequency_words": get_low_freq_words(corrected),
    }

In [10]:
test_input = (
    "න්ක ඔන් එකෙතියෙන්නේ කපල නෑ පියෝටීවිසහ ඔයිස් දෙක මෝන්න කි තියේන "
    "සමොකද්වෙලා තියෙන්නේ වැඩකර පියෝටීවිකත් වැඩකරනා ටවඉස් එකත් හරිමේ "
    "ලයින් එකේ ඒ පහසු කන්දක මෝන් කෙසර තියෙනන ගැටලුවක් නෑ ලයින් "
    "රනාඩෙදාස් පන්සේ අසුතුනත්ත සර්ගෙවන්නතියනැච්චර ගවන්න තියෙන්නේවසසෙ "
    "පොඩ්ඩක් බලන්න කියන මේ සමහරක් ගලට මේ කේබල් වහිනකොට ගලවනානේ "
    "අරකළුපාඩ බොක්සකින් නවා නිට් ගේලක තියෙනන හතරෙන්නේ කවුලලිව පොඩ්ඩක් "
    "හමට ගැලිල ඇති පශික්කරනකන් මේ පත්තනං ඔව කෙස හරිසහරිවකස එසටමත "
    "සවදස් සබදාසක් ස ඇමතිමකිසනා රඳඉන්න"
)

SEP = "=" * 65
result = correct_sinhala_text(test_input)

print(f"\n{SEP}")
print("INPUT:")
print(result["input"])
print(f"\n{SEP}")
print("CORRECTED OUTPUT:")
print(result["corrected_text"])
print(f"\n{SEP}")
print("UNKNOWN WORDS :", result["unknown_words"])
print("LOW FREQ WORDS:", result["low_frequency_words"])

▶ Pre-processed input ready.
▶ Pass 1: Full correction…
  ✓ Pass 1 done.
▶ Pass 2: Refinement…
  ✓ Pass 2 done.
▶ Pass 3: Fixing unknown words: ['ඇමතුමේ', 'කරන්නකෝ', 'ගේමක', 'වහිනකොට']
  ✓ Pass 3 done.

INPUT:
න්ක ඔන් එකෙතියෙන්නේ කපල නෑ පියෝටීවිසහ ඔයිස් දෙක මෝන්න කි තියේන සමොකද්වෙලා තියෙන්නේ වැඩකර පියෝටීවිකත් වැඩකරනා ටවඉස් එකත් හරිමේ ලයින් එකේ ඒ පහසු කන්දක මෝන් කෙසර තියෙනන ගැටලුවක් නෑ ලයින් රනාඩෙදාස් පන්සේ අසුතුනත්ත සර්ගෙවන්නතියනැච්චර ගවන්න තියෙන්නේවසසෙ පොඩ්ඩක් බලන්න කියන මේ සමහරක් ගලට මේ කේබල් වහිනකොට ගලවනානේ අරකළුපාඩ බොක්සකින් නවා නිට් ගේලක තියෙනන හතරෙන්නේ කවුලලිව පොඩ්ඩක් හමට ගැලිල ඇති පශික්කරනකන් මේ පත්තනං ඔව කෙස හරිසහරිවකස එසටමත සවදස් සබදාසක් ස ඇමතිමකිසනා රඳඉන්න

CORRECTED OUTPUT:
ලයින් ඔන් එකේ තියෙන්නේ, කපලා නෑ. පියෝ ටීවී සහ වොයිස් දෙක ඔන් එකේ තියෙන්නේ, සම්බන්ධ වෙලා තියෙන්නේ, වැඩ කරනවා. පියෝ ටීවී එකත් වැඩ කරනවා, වොයිස් එකත් හරි. මේ ලයින් එකේ ප්‍රධාන කොටසක කිසිම ගැටලුවක් නෑ. රුපියල් දෙදහස් පන්සිය අසූ තුනක් සර් ගෙවන්න තියෙනවා. ඊට අමතරව, පොඩ්ඩක් බලන්න කියන්නේ, මේ සමහරක් කේබල් වහිනකොට